# Финал

Беру те же признаки, модели и blend, которые проверяла в предыдущем ноутбуке, обучаюсь на всем train и делаю submission

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 2026

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features_base import (
    filter_events_by_window,
    prepare_events,
    build_base_features,
)

from src.features_cross import (
    build_cross_cookie_features,
    percentile_rank,
)

print(PROJECT_ROOT)


/Users/an.m.titova/Documents/bot_detection_case


In [2]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

events = pd.read_csv(
    DATA_DIR / "events.csv.gz",
    parse_dates=["event_ts"],
)

sample_submission = pd.read_csv(
    PROJECT_ROOT / "sample_submission.csv"
)

print(train.shape, test.shape, events.shape)
sample_submission.head()


(11091, 5) (4909, 4) (328905, 14)


,cookie_id,score
0,ck_99a4e5ef02d89493,0.5
1,ck_54d463f7fd89ec3d,0.5
2,ck_0fbbc7a6af300368,0.5
3,ck_8fd4937eadd64fb6,0.5
4,ck_dbb4bd4eda97255d,0.5


In [3]:
events_train = filter_events_by_window(
    events,
    train,
)

events_test = filter_events_by_window(
    events,
    test,
)

events_train = prepare_events(events_train)
events_test = prepare_events(events_test)

print(len(events_train), len(events_test))


198436 89690


In [4]:
EVENT_TYPES = sorted(
    events_train["event_name"]
    .dropna()
    .unique()
)

PLATFORM_TYPES = sorted(
    events_train["platform_clean"]
    .dropna()
    .unique()
)

print(EVENT_TYPES)
print(PLATFORM_TYPES)


['contact_chat_open', 'contact_message_sent', 'contact_phone_show', 'favorite_add', 'item_view', 'login', 'photo_swipe', 'search_results_view', 'seller_page_view']
['android', 'desktop', 'ios', 'iphone', 'web']


In [5]:
train_base = build_base_features(
    train,
    events_train,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

test_base = build_base_features(
    test,
    events_test,
    EVENT_TYPES,
    PLATFORM_TYPES,
)

train_base = train_base.merge(
    train[["cookie_id", "target"]],
    on="cookie_id",
    how="left",
)

print(train_base.shape)
print(test_base.shape)


/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:489: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"share_platform_{platform}"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:879: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["events_per_active_hour"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:890: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

(11091, 195)
(4909, 194)


/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:879: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["events_per_active_hour"] = (
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:890: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features["unique_items_per_item_view"] = safe_ratio(
/Users/an.m.titova/Documents/bot_detection_case/src/features_base.py:899: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has 

Для train cross-cookie частоты считаю по всему train с leave-one-cookie-out
Для test reference только train. Сам test между собой не использую

In [6]:
train_cross = build_cross_cookie_features(
    reference_events=events_train,
    target_events=events_train,
    target_cookie_ids=train["cookie_id"],
    leave_one_cookie_out=True,
)

test_cross = build_cross_cookie_features(
    reference_events=events_train,
    target_events=events_test,
    target_cookie_ids=test["cookie_id"],
    leave_one_cookie_out=False,
)

print(train_cross.shape)
print(test_cross.shape)


(11091, 61)
(4909, 61)


In [7]:
train_final = train_base.merge(
    train_cross,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

test_final = test_base.merge(
    test_cross,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

NON_FEATURE_COLS = [
    "cookie_id",
    "target",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

feature_cols = [
    col
    for col in train_final.columns
    if col not in NON_FEATURE_COLS
]

test_feature_cols = [
    col
    for col in test_final.columns
    if col not in [
        "cookie_id",
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ]
]

assert feature_cols == test_feature_cols
assert train_final["cookie_id"].is_unique
assert test_final["cookie_id"].is_unique

print("features:", len(feature_cols))
print(train_final.shape, test_final.shape)


features: 250
(11091, 255) (4909, 254)


In [8]:
X_train = train_final[feature_cols]
y_train = train_final["target"]

X_test = test_final[feature_cols]


In [9]:
hist = HistGradientBoostingClassifier(
    learning_rate=0.045,
    max_iter=350,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

hist.fit(
    X_train,
    y_train,
)

hist_score = hist.predict_proba(X_test)[:, 1]

hist_score[:5]


array([0.02465798, 0.25386193, 0.1236574 , 0.01875454, 0.24376725])

In [10]:
cat = CatBoostClassifier(
    iterations=460,
    depth=7,
    learning_rate=0.03,
    l2_leaf_reg=8.0,
    class_weights=[1.0, 4.0],
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    random_strength=1.0,
    bagging_temperature=1.0,
    border_count=128,
    allow_writing_files=False,
    verbose=False,
)

cat.fit(
    X_train,
    y_train,
)

cat_score = cat.predict_proba(X_test)[:, 1]

cat_score[:5]


array([0.00178509, 0.30840899, 0.07627611, 0.00482074, 0.12563471])

Веса не подбираю по test. Оставляю 0.55 для HGB и 0.45 для CatBoost, как было в валидации

In [11]:
final_score = (
    0.55 * percentile_rank(hist_score)
    + 0.45 * percentile_rank(cat_score)
)

pd.Series(final_score).describe()


count    4909.000000
mean        0.500102
std         0.279528
min         0.000886
25%         0.264321
50%         0.495274
75%         0.730892
max         0.997703
dtype: float64

Score оставляю непрерывным. Никаких 0/1 здесь не делаю, потому что проверяющая метрика сама работает с порогом (из задания)

In [12]:
scores = pd.DataFrame({
    "cookie_id": test_final["cookie_id"],
    "score": final_score,
})

submission = (
    sample_submission[["cookie_id"]]
    .merge(
        scores,
        on="cookie_id",
        how="left",
        validate="one_to_one",
    )
)

assert list(submission.columns) == [
    "cookie_id",
    "score",
]

assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert set(submission["cookie_id"]) == set(test["cookie_id"])
assert submission["score"].notna().all()
assert submission["score"].between(0, 1).all()

submission.head()


,cookie_id,score
0,ck_99a4e5ef02d89493,0.698207
1,ck_54d463f7fd89ec3d,0.596598
2,ck_0fbbc7a6af300368,0.163750
3,ck_8fd4937eadd64fb6,0.143492
4,ck_dbb4bd4eda97255d,0.350377


In [13]:
submission.to_csv(
    PROJECT_ROOT / "submission.csv",
    index=False,
)

print("saved:", PROJECT_ROOT / "submission.csv")
print("rows:", len(submission))
print(
    "score range:",
    submission["score"].min(),
    submission["score"].max(),
)


saved: /Users/an.m.titova/Documents/bot_detection_case/submission.csv
rows: 4909
score range: 0.0008861275208800164 0.9977031982073743


In [14]:
pd.read_csv(
    PROJECT_ROOT / "submission.csv"
).head()


,cookie_id,score
0,ck_99a4e5ef02d89493,0.698207
1,ck_54d463f7fd89ec3d,0.596598
2,ck_0fbbc7a6af300368,0.163750
3,ck_8fd4937eadd64fb6,0.143492
4,ck_dbb4bd4eda97255d,0.350377
